## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-24-09-52-00 +0000,wsj,German Business Sentiment Rises Again as Firms...,https://www.wsj.com/economy/central-banking/ge...
1,2026-11-24-09-52-00 +0000,wsj,"Jensen Huang’s Return Sparks Rally in Asia, bu...",https://www.wsj.com/finance/stocks/asian-equit...
2,2026-11-24-09-45-00 +0000,wsj,Companies tied to an investment firm affiliate...,https://www.wsj.com/politics/policy/democrats-...
3,2026-11-24-09-42-54 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...
4,2026-11-24-09-39-00 +0000,wsj,Sanofi Strikes $41 Billion Deal to Buy U.S. Dr...,https://www.wsj.com/business/deals/sanofi-to-b...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2716/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
9,huang,58
38,nvidia,47
22,trump,46
8,jensen,44
10,return,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
18,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...,273
3,2026-11-24-09-42-54 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,236
283,2026-11-23-15-53-43 +0000,nypost,Trump suggests Jensen Huang could have moved t...,https://nypost.com/2026/11/23/us-news/trump-su...,233
281,2026-11-23-16-12-26 +0000,nypost,Trump assured Nvidia board Jensen Huang unlike...,https://nypost.com/2026/11/23/us-news/trump-as...,232
109,2026-11-23-23-46-05 +0000,nypost,Five wild Trump quotes about reinstated Nvidia...,https://nypost.com/2026/11/23/us-news/five-wil...,232


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
18,273,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...
140,88,2026-11-23-22-29-00 +0000,wsj,"Rep. Jimmy Gomez (D., Calif.) said he would re...",https://www.wsj.com/politics/democratic-lawmak...
56,58,2026-11-24-03-30-00 +0000,nypost,China’s second-richest man’s deal to drain Cal...,https://nypost.com/2026/11/23/us-news/chinese-...
375,54,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...
344,47,2026-11-23-12-10-26 +0000,chitri,Ukrainian drones hit Russian warship at naval ...,https://www.chicagotribune.com/2026/11/23/ukra...
200,47,2026-11-23-20-30-16 +0000,nypost,Russia has lost astonishing 700K soldiers sinc...,https://nypost.com/2026/11/23/world-news/russi...
49,42,2026-11-24-04-15-00 +0000,wsj,A federal appeals court has cleared the way fo...,https://www.wsj.com/politics/national-security...
55,40,2026-11-24-03-38-00 +0000,wsj,Ilhan Omar Investigation Covers Most of Her Li...,https://www.wsj.com/politics/policy/ilhan-omar...
180,39,2026-11-23-21-24-51 +0000,nypost,Nasdaq surges 6.5% in second-largest one-day g...,https://nypost.com/2026/11/23/business/nasdaq-...
105,34,2026-11-24-00-00-00 +0000,wsj,U.S. Eyes Iran as Possible Culprit in Dozens o...,https://www.wsj.com/politics/national-security...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
